In [ ]:
from langchain.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
import os
import json

llm = ChatOpenAI(
    model=os.environ["MODEL_NAME"],
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

parser = StructuredOutputParser.from_response_schemas([
    ResponseSchema(name='a', description='用户问题的答案，格式是以英文逗号隔开的中文字符串'),
    ResponseSchema(name='b', description='你回答用户问题所依据的答案'),
    ResponseSchema(name='c', description='问题答案的可信度评分，格式是百分数')
])

prompt = PromptTemplate.from_template(
    '''
    列出3个 {country} 的著名的互联网公司。
    {instructions}
    '''
)

chain = prompt | llm | parser

rst = chain.invoke({'country': '中国', 'instructions': parser.get_format_instructions()})
print(json.dumps(rst, indent=2, ensure_ascii=False))

In [ ]:
from langchain.output_parsers import CommaSeparatedListOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
import os

llm = ChatOpenAI(
    model=os.environ["MODEL_NAME"],
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

parser = CommaSeparatedListOutputParser()

prompt = PromptTemplate.from_template(
    '''
     列出 {country} 排名前5的互联网公司。
    {instructions}
    '''
)

chain = prompt | llm | parser
rst = chain.invoke({'country': '中国', 'instructions': parser.get_format_instructions()})
print(rst)

In [ ]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    model=os.environ["MODEL_NAME"],
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

parser1 = lambda x: x.content + '🔥🔥🔥'
parser2 = lambda x: x + '🚀🚀🚀'

chain = llm | parser1 | parser2

rst = chain.invoke('你好')

print(rst)


In [80]:
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough, RunnableAssign, RunnableSequence, RunnableLambda, RunnableParallel, RunnableBranch
import os

llm = ChatOpenAI(
    model=os.environ["MODEL_NAME"],
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

parser1 = lambda x: x.content + '🔥🔥🔥'
parser2 = lambda x: x + '🚀🚀🚀'

def print_parser(x):
    print(x)
    return x


chain = RunnableSequence(
    first=RunnableLambda(print_parser),
    middle=[
        RunnableParallel({ 'input': print_parser }),
        RunnableLambda(lambda x: x['input']),
        llm, 
        RunnableLambda(parser1)
    ],
    last=RunnableLambda(parser2)
)

# rst = chain.invoke('你好')
# print(rst)

# RunnableLambda(parser2).invoke('你好')

# branch = RunnableBranch(
#     (lambda x: x == '你好', lambda x: x + '🔥🔥🔥'),
#     (lambda x: x == '再见', lambda x: x + '🚀🚀🚀'),
#     lambda x: x + '🌳🌳🌳'
# )

# rst = branch.invoke('哈哈')

# print(rst)

# assign = RunnableAssign({ 'context': lambda x: f"这里是上下文：{x['input']}", 'history': lambda x: '历史记录' })
# assign.invoke({'input': '你好'})

# assign = RunnablePassthrough.assign(
#     context=lambda x: f"这里是上下文：{x['input']}",
#     history=lambda x: '历史记录'
# )

# assign.invoke({ 'input': '你好' })

def func(x):
    print(x + '🌳🌳🌳')
    return x + '🔥🔥🔥'

through = RunnablePassthrough(func=func)

rst = through.invoke('你好')

print(rst)



你好🌳🌳🌳
你好
